In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
os.chdir('/content/drive/MyDrive/network-intrusion-ml')
sys.path.append('/content/drive/MyDrive/network-intrusion-ml')

import numpy as np
import pandas as pd
import joblib

from src.utils import ROOT, PROCESSED_DIR, METRIC_DIR, MODEL_DIR, RANDOM_STATE

with open(METRIC_DIR / "label_mapping.json") as f:
    label_mapping = json.load(f)
with open(MODEL_DIR / "model_features.json") as f:
    FEATURES = json.load(f)

APP_DIR = ROOT / "app"
APP_DIR.mkdir(exist_ok=True)
print("Project root:", ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/network-intrusion-ml


In [2]:
test = pd.read_parquet(PROCESSED_DIR / "test.parquet", columns=FEATURES + ["Label_Group"])
y_t = test["Label_Group"].map(label_mapping).values
yb = (y_t != 0).astype(int)
pred_b = np.load(METRIC_DIR / "07_test_pred_binary.npy")

rng = np.random.default_rng(RANDOM_STATE)
idx = []
for c in range(len(label_mapping)):
    ids = np.where(y_t == c)[0]
    idx.extend(rng.choice(ids, size=min(15, len(ids)), replace=False))
fa = np.where((yb == 0) & (pred_b == 1))[0]
miss = np.where((yb == 1) & (pred_b == 0))[0]
idx.extend(rng.choice(fa, 3, replace=False))
idx.extend(rng.choice(miss, 3, replace=False))
idx = list(dict.fromkeys(int(i) for i in idx))

demo = test.iloc[idx].rename(columns={"Label_Group": "true_label"}).reset_index(drop=True)
demo.to_csv(APP_DIR / "demo_flows.csv", index=False)
print("Saved demo_flows.csv:", demo.shape)
print(demo["true_label"].value_counts().to_string())

# Round-trip check: models + CSV must reproduce the earlier test predictions
bin_model = joblib.load(MODEL_DIR / "binary_model.pkl")
back = pd.read_csv(APP_DIR / "demo_flows.csv")
p = (bin_model.predict_proba(back[FEATURES])[:, 1] >= 0.5).astype(int)
print(f"\nMatches the saved test predictions: {(p == pred_b[idx]).mean()*100:.1f}%  (expect 100%)")
print("Model file sizes (MB):", {f.name: round(f.stat().st_size / 1e6, 1) for f in MODEL_DIR.glob("*.pkl")})

Saved demo_flows.csv: (135, 49)
true_label
BENIGN          18
Bot             17
PortScan        16
DDoS            15
FTP-Patator     15
DoS             15
Web Attack      15
SSH-Patator     15
Infiltration     7
Heartbleed       2

Matches the saved test predictions: 100.0%  (expect 100%)
Model file sizes (MB): {'binary_model.pkl': 0.4, 'multiclass_model.pkl': 2.6}


In [3]:
import sys, sklearn, xgboost, shap, matplotlib

req = f"""streamlit>=1.30
pandas=={pd.__version__}
numpy=={np.__version__}
scikit-learn=={sklearn.__version__}
xgboost=={xgboost.__version__}
shap=={shap.__version__}
joblib=={joblib.__version__}
matplotlib=={matplotlib.__version__}
"""
(ROOT / "requirements.txt").write_text(req)
print(req)
print("Colab Python version:", sys.version.split()[0])

streamlit>=1.30
pandas==2.2.3
numpy==2.4.6
scikit-learn==1.6.1
xgboost==3.4.1
shap==0.54.1
joblib==1.6.0
matplotlib==3.10.0

Colab Python version: 3.13.18


In [4]:
%%writefile /content/drive/MyDrive/network-intrusion-ml/app/app.py
"""Network Intrusion Detection demo (Streamlit).
Run from the project folder:  streamlit run app/app.py
"""
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import streamlit as st

ROOT = Path(__file__).resolve().parent.parent
sys.path.append(str(ROOT))                      # lets joblib find src/preprocessing.py
from src.preprocessing import signed_log1p      # noqa: F401  (needed to load the saved models)

st.set_page_config(page_title="Network Intrusion Detector", layout="wide")


@st.cache_resource
def load_assets():
    bin_model = joblib.load(ROOT / "models" / "binary_model.pkl")
    multi_model = joblib.load(ROOT / "models" / "multiclass_model.pkl")
    with open(ROOT / "models" / "model_features.json") as f:
        features = json.load(f)
    with open(ROOT / "models" / "label_mapping.json") as f:
        mapping = json.load(f)
    class_names = [c for c, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
    explainer = shap.TreeExplainer(bin_model.named_steps["model"])
    return bin_model, multi_model, features, class_names, explainer


bin_model, multi_model, FEATURES, CLASS_NAMES, explainer = load_assets()


def predict(df, threshold):
    """Binary model decides ATTACK/BENIGN; the 10-class model names the attack type."""
    X = df[FEATURES]
    p_attack = bin_model.predict_proba(X)[:, 1]
    p_multi = multi_model.predict_proba(X)
    attack_part = p_multi[:, 1:]
    attack_norm = attack_part / (attack_part.sum(axis=1, keepdims=True) + 1e-12)
    attack_type = np.array(CLASS_NAMES[1:])[attack_norm.argmax(axis=1)]
    is_attack = p_attack >= threshold
    return pd.DataFrame({
        "verdict": np.where(is_attack, "ATTACK", "BENIGN"),
        "attack_probability": p_attack.round(4),
        "likely_attack_type": np.where(is_attack, attack_type, "-"),
        "type_confidence": np.where(is_attack, attack_norm.max(axis=1).round(3), np.nan),
    })


def explain(row_df, top_n=8):
    """Which clues pushed this flow toward ATTACK (positive) or BENIGN (negative)."""
    x_log = bin_model[:-1].transform(row_df[FEATURES])
    sv = explainer.shap_values(x_log)[0]
    table = pd.DataFrame({"clue": FEATURES,
                          "raw value": row_df[FEATURES].iloc[0].values,
                          "push": sv})
    order = table["push"].abs().sort_values(ascending=False).index
    return table.loc[order].head(top_n).reset_index(drop=True)


def show_explanation(row_df):
    table = explain(row_df)
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(t["clue"], t["push"], color=np.where(t["push"] > 0, "#d62728", "#1f77b4").tolist())
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("push toward ATTACK (right, red)   |   toward BENIGN (left, blue)")
    fig.tight_layout()
    left, right = st.columns([3, 2])
    left.pyplot(fig)
    plt.close(fig)
    right.dataframe(table.round(3))
    st.caption("Why this verdict? The bars come from SHAP applied to the binary model. "
               "Longer bar = that clue mattered more for this flow.")


# ------------------------------------------------------------------ page
st.title("Network Intrusion Detector")
st.caption("Explainable ML demo on CIC-IDS2017. A binary XGBoost model decides ATTACK vs BENIGN; "
           "a 10-class XGBoost model names the attack type. Educational project, not a production IDS.")

mode = st.sidebar.radio("Input", ["Try a demo flow", "Upload a CSV of flows"])
threshold = st.sidebar.slider("Attack alert threshold", 0.05, 0.95, 0.50, 0.05,
                              help="The models were evaluated at 0.50. Lower = fewer missed attacks but more false alarms.")

if mode == "Try a demo flow":
    demo = pd.read_csv(ROOT / "app" / "demo_flows.csv")
    i = st.selectbox("Choose a network flow", list(range(len(demo))), format_func=lambda k: f"Flow #{k}")
    row = demo.iloc[[i]]
    res = predict(row, threshold).iloc[0]
    c1, c2, c3 = st.columns(3)
    c1.metric("Verdict", res["verdict"])
    c2.metric("Attack probability", f"{res['attack_probability']:.3f}")
    c3.metric("Likely attack type", res["likely_attack_type"])
    st.info(f"Actual label in the dataset: **{demo.loc[i, 'true_label']}**")
    show_explanation(row)
else:
    st.write("Upload a CSV with one flow per row. It must contain these 48 columns "
             "(CIC-IDS2017 column names; extra columns are ignored).")
    with st.expander("Show the required columns"):
        st.write(FEATURES)
    up = st.file_uploader("CSV file", type="csv")
    if up is not None:
        df = pd.read_csv(up)
        df.columns = df.columns.str.strip()
        missing = [c for c in FEATURES if c not in df.columns]
        if missing:
            st.error(f"Missing columns: {missing}")
            st.stop()
        df = df.replace([np.inf, -np.inf], np.nan)
        res = predict(df, threshold)
        st.subheader("Summary")
        a, b = st.columns(2)
        a.metric("Flows checked", f"{len(res):,}")
        b.metric("Flagged as attack", f"{(res['verdict'] == 'ATTACK').sum():,}")
        st.write(res["likely_attack_type"][res["verdict"] == "ATTACK"].value_counts().rename("count"))
        st.subheader("Results (first 1,000 rows)")
        st.dataframe(res.head(1000))
        st.download_button("Download all results (CSV)", res.to_csv(index=False), "predictions.csv")
        k = st.number_input("Row to explain", 0, len(df) - 1, 0)
        show_explanation(df.iloc[[int(k)]])

with st.expander("How well does it work? (locked test set, 499,596 flows)"):
    for name in ["07_test_binary_metrics.csv", "07_test_multiclass_per_class.csv"]:
        p = ROOT / "results" / "tables" / name
        if p.exists():
            st.write(name.replace(".csv", ""))
            st.dataframe(pd.read_csv(p))
    st.write("Known limits: the data comes from one lab testbed; the model leans on destination port and "
             "TCP window size; Heartbleed and Infiltration have only 2 and 7 test flows.")

Writing /content/drive/MyDrive/network-intrusion-ml/app/app.py


In [5]:
%%writefile /content/drive/MyDrive/network-intrusion-ml/README.md
# Explainable ML for Network Intrusion Detection and Attack Classification

Course project (SRM University-AP). Detects malicious network flows in the CIC-IDS2017 dataset
(binary: BENIGN vs ATTACK) and names the attack type (10 classes), with SHAP explanations
and a Streamlit demo.

## Results (locked test set, 499,596 flows, scored once)

| Task | Result |
|---|---|
| Binary (XGBoost) | recall 0.9986, precision 0.9968, F1 0.9977, ROC-AUC 0.99997; 116 missed attacks, 275 false alarms (0.066% of normal flows) |
| 10-class (XGBoost + SMOTE) | accuracy 0.9992, macro-F1 0.9566; weakest class: Bot (F1 0.84) |

Heartbleed (2 test flows) and Infiltration (7 test flows) are too small for reliable estimates.

## Method in short

1. Cleaning: removed NaN/inf rows, 329,896 duplicate rows, 1 duplicate and 8 constant columns; 15 raw labels grouped into 10 classes.
2. Stratified 80/20 split; the test set stayed locked until the final evaluation.
3. Six models compared with stratified 5-fold cross-validation (mean +/- std); signed-log transform + scaling for linear/KNN/SVM models.
4. Class imbalance: class weights, undersampling and SMOTE compared; SMOTE used for the 10-class model (training folds only).
5. Randomized hyper-parameter search (little gain, models near their ceiling).
6. Feature sets compared: 48 correlation-filtered features kept (the 25-feature set failed on rare classes).
7. Explainability: XGBoost gain importance and SHAP (global, per-class, single-flow).

## Limitations (please read)

- One lab testbed (CIC-IDS2017); results may not transfer to other networks.
- SHAP shows reliance on destination port and TCP window size (Init_Win_bytes_*). Removing both lowers binary F1 to about 0.990 and Bot F1 to about 0.71 in cross-validation.
- Most Web Attack flows have zero payload bytes, so that class is probably recognised by flow/lab characteristics, not by attack content.
- Random split: flows from the same attack sessions appear in train and test.

## Run the demo (Windows)

    python -m venv venv
    venv\Scripts\activate
    pip install -r requirements.txt
    streamlit run app/app.py

## Folders

- `notebooks/` - all experiments in order (01 to 09)
- `src/` - helper code (`preprocessing.py` is needed to load the saved models)
- `models/` - final binary and 10-class pipelines (joblib)
- `results/` - tables, figures, experiment log
- `app/` - Streamlit app and a small demo sample

The dataset is not included. Download CIC-IDS2017 from the Canadian Institute for Cybersecurity website.

Writing /content/drive/MyDrive/network-intrusion-ml/README.md
